In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="white")

In [4]:
import random
np.random.seed(42)

num_recursos = 20000 

formatos = ['PDF', 'Video', 'Foro', 'Repositorio', 'Podcast', 'Laboratorio Virtual', 'Artículo']
carreras = [
    'Ciencias de la Computación', 'Ingeniería Industrial', 'Negocios', 
    'Medicina', 'Diseño', 'Economía', 'Psicología', 'Ingeniería Mecatrónica'
]
etiquetas_posibles = [
    'Data Science', 'Backend', 'Matemáticas', 'Finanzas', 'Algoritmos', 
    'Bases de Datos', 'Arquitectura Mainframe', 'Machine Learning', 'UX/UI', 
    'Hardware', 'COBOL', 'Ensamblador', 'Criptografía', 'Monitoreo IoT', 
    'Desarrollo de Videojuegos', 'Estadística', 'Redes'
]
niveles = ['Básico', 'Intermedio', 'Avanzado']

ids = np.arange(1, num_recursos + 1)
titulos = [f"Material_Estudio_{i}" for i in range(1, num_recursos + 1)]
formato_col = np.random.choice(formatos, num_recursos)
carrera_col = np.random.choice(carreras, num_recursos)
nivel_col = np.random.choice(niveles, num_recursos)

etiquetas_col = [", ".join(np.random.choice(etiquetas_posibles, size=np.random.randint(1, 5), replace=False)) for _ in range(num_recursos)]

df_catalogo = pd.DataFrame({
    'ID_Recurso': ids,
    'Titulo': titulos,
    'Formato': formato_col,
    'Carrera': carrera_col,
    'Nivel_Dificultad': nivel_col,
    'Etiquetas': etiquetas_col
})

print(f"Total de recursos: {df_catalogo.shape[0]}")
df_catalogo.head()

Total de recursos: 20000


,ID_Recurso,Titulo,Formato,Carrera,Nivel_Dificultad,Etiquetas
0,1,Material_Estudio_1,Artículo,Ingeniería Mecatrónica,Avanzado,"UX/UI, Bases de Datos"
1,2,Material_Estudio_2,Repositorio,Ciencias de la Computación,Básico,"Estadística, COBOL, Redes"
2,3,Material_Estudio_3,Podcast,Diseño,Intermedio,"Hardware, Arquitectura Mainframe, UX/UI"
3,4,Material_Estudio_4,Artículo,Medicina,Básico,Bases de Datos
4,5,Material_Estudio_5,Foro,Negocios,Básico,"Criptografía, Ensamblador, Data Science"


In [7]:
df_categorico = pd.get_dummies(df_catalogo[['Formato', 'Carrera', 'Nivel_Dificultad']], dtype=int)

df_etiquetas = df_catalogo['Etiquetas'].str.get_dummies(sep=', ')

matriz_caracteristicas = pd.concat([df_categorico, df_etiquetas], axis=1)

matriz_caracteristicas.index = df_catalogo['ID_Recurso']

matriz_caracteristicas.head()

,Formato_Artículo,Formato_Foro,Formato_Laboratorio Virtual,Formato_PDF,Formato_Podcast,Formato_Repositorio,Formato_Video,Carrera_Ciencias de la Computación,Carrera_Diseño,Carrera_Economía,...,Desarrollo de Videojuegos,Ensamblador,Estadística,Finanzas,Hardware,Machine Learning,Matemáticas,Monitoreo IoT,Redes,UX/UI
ID_Recurso,,,,,,,,,,,,,,,,,,,,,
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,1,0,1,0,0,...,0,0,1,0,0,0,0,0,1,0
3,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,1
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def recomendar_recursos(id_recurso, top_n=5):
    vector_recurso_actual = matriz_caracteristicas.loc[[id_recurso]]
    
    similitudes = cosine_similarity(vector_recurso_actual, matriz_caracteristicas)[0]
    df_catalogo['Puntuacion_Similitud'] = similitudes
    
    recomendaciones = df_catalogo[df_catalogo['ID_Recurso'] != id_recurso].sort_values(
        by='Puntuacion_Similitud', ascending=False
    ).head(top_n)
    
    columnas_reporte = ['ID_Recurso', 'Titulo', 'Formato', 'Carrera', 'Nivel_Dificultad', 'Etiquetas', 'Puntuacion_Similitud']
    
    return recomendaciones[columnas_reporte]

recurso_prueba = 15 

print(f"--- RECURSO ACTUAL VIENDO EL ESTUDIANTE (ID: {recurso_prueba}) ---")
print(df_catalogo[df_catalogo['ID_Recurso'] == recurso_prueba][['Titulo', 'Formato', 'Carrera', 'Nivel_Dificultad', 'Etiquetas']].to_string(index=False))

print("\n--- TOP 5 RECOMENDACIONES DEL ALGORITMO ---")
recomendaciones_finales = recomendar_recursos(recurso_prueba, top_n=5)
print(recomendaciones_finales.to_string(index=False))

--- RECURSO ACTUAL VIENDO EL ESTUDIANTE (ID: 15) ---
             Titulo     Formato Carrera Nivel_Dificultad                                         Etiquetas
Material_Estudio_15 Repositorio  Diseño         Avanzado Criptografía, Finanzas, Desarrollo de Videojuegos

--- TOP 5 RECOMENDACIONES DEL ALGORITMO ---
 ID_Recurso                 Titulo     Formato Carrera Nivel_Dificultad                                         Etiquetas  Puntuacion_Similitud
      19209 Material_Estudio_19209 Repositorio  Diseño         Avanzado                     Redes, Criptografía, Finanzas              0.833333
       6052  Material_Estudio_6052 Repositorio  Diseño         Avanzado Data Science, Finanzas, Desarrollo de Videojuegos              0.833333
       2475  Material_Estudio_2475 Repositorio  Diseño         Avanzado        Redes, Finanzas, Desarrollo de Videojuegos              0.833333
      12583 Material_Estudio_12583 Repositorio  Diseño         Avanzado                                         

In [12]:
peso_carrera = 4.0      # El más crítico
peso_etiquetas = 3.0    # El tema central del recurso
peso_dificultad = 1.5   # Importante, pero flexible
peso_formato = 0.5      # Menos importante

matriz_pesada = matriz_caracteristicas.copy()

cols_carrera = [col for col in matriz_pesada.columns if col.startswith('Carrera_')]
matriz_pesada[cols_carrera] *= peso_carrera

cols_dificultad = [col for col in matriz_pesada.columns if col.startswith('Nivel_Dificultad_')]
matriz_pesada[cols_dificultad] *= peso_dificultad

cols_formato = [col for col in matriz_pesada.columns if col.startswith('Formato_')]
matriz_pesada[cols_formato] *= peso_formato

cols_etiquetas = [col for col in matriz_pesada.columns if col not in cols_carrera + cols_dificultad + cols_formato]
matriz_pesada[cols_etiquetas] *= peso_etiquetas

def recomendar_recursos_pesados(id_recurso, top_n=5):
    vector_recurso = matriz_pesada.loc[[id_recurso]]
    
    similitudes = cosine_similarity(vector_recurso, matriz_pesada)[0]
    
    df_catalogo['Puntuacion_Similitud'] = similitudes
    
    recomendaciones = df_catalogo[df_catalogo['ID_Recurso'] != id_recurso].sort_values(
        by='Puntuacion_Similitud', ascending=False
    ).head(top_n)
    
    return recomendaciones[['ID_Recurso', 'Titulo', 'Formato', 'Carrera', 'Nivel_Dificultad', 'Etiquetas', 'Puntuacion_Similitud']]

recurso_prueba = 15 

print(f"--- RECURSO ACTUAL VIENDO EL ESTUDIANTE (ID: {recurso_prueba}) ---")
print(df_catalogo[df_catalogo['ID_Recurso'] == recurso_prueba][['Titulo', 'Formato', 'Carrera', 'Nivel_Dificultad', 'Etiquetas']].to_string(index=False))

print("\n--- TOP 5 RECOMENDACIONES (ALGORITMO CON PESOS PERSONALIZADOS) ---")
recomendaciones_pesadas = recomendar_recursos_pesados(recurso_prueba, top_n=5)
print(recomendaciones_pesadas.to_string(index=False))

--- RECURSO ACTUAL VIENDO EL ESTUDIANTE (ID: 15) ---
             Titulo     Formato Carrera Nivel_Dificultad                                         Etiquetas
Material_Estudio_15 Repositorio  Diseño         Avanzado Criptografía, Finanzas, Desarrollo de Videojuegos

--- TOP 5 RECOMENDACIONES (ALGORITMO CON PESOS PERSONALIZADOS) ---
 ID_Recurso                 Titulo             Formato Carrera Nivel_Dificultad                                                Etiquetas  Puntuacion_Similitud
       3639  Material_Estudio_3639 Laboratorio Virtual  Diseño         Avanzado Finanzas, Desarrollo de Videojuegos, Criptografía, COBOL              0.908688
       6398  Material_Estudio_6398            Artículo  Diseño         Avanzado                  Desarrollo de Videojuegos, Criptografía              0.889520
        533   Material_Estudio_533            Artículo  Diseño         Avanzado                                   Criptografía, Finanzas              0.889520
      17639 Material_Estudio_

In [14]:
def recomendar_recursos_hibridos(id_recurso, top_n=5):
    vector_recurso = matriz_pesada.loc[[id_recurso]]
    similitudes = cosine_similarity(vector_recurso, matriz_pesada)[0]
    
    df_catalogo['Similitud_Pura'] = similitudes
    
    df_catalogo['Rating_Normalizado'] = df_catalogo['Valoracion_Estudiantes'] / 5.0
    
    df_catalogo['Score_Final'] = (df_catalogo['Similitud_Pura'] * 0.7) + (df_catalogo['Rating_Normalizado'] * 0.3)
    
    recomendaciones = df_catalogo[df_catalogo['ID_Recurso'] != id_recurso].sort_values(
        by='Score_Final', ascending=False
    ).head(top_n)
    
    columnas_finales = ['ID_Recurso', 'Titulo', 'Formato', 'Carrera', 'Etiquetas', 'Valoracion_Estudiantes', 'Score_Final']
    return recomendaciones[columnas_finales]

recurso_prueba = 15 

print(f"--- RECURSO ACTUAL (ID: {recurso_prueba}) ---")
print(df_catalogo[df_catalogo['ID_Recurso'] == recurso_prueba][['Titulo', 'Formato', 'Carrera', 'Valoracion_Estudiantes', 'Etiquetas']].to_string(index=False))

print("\n--- TOP 5 RECOMENDACIONES (SIMILITUD + CALIDAD) ---")
recomendaciones_hibridas = recomendar_recursos_hibridos(recurso_prueba, top_n=5)
print(recomendaciones_hibridas.to_string(index=False))

--- RECURSO ACTUAL (ID: 15) ---
             Titulo     Formato Carrera  Valoracion_Estudiantes                                         Etiquetas
Material_Estudio_15 Repositorio  Diseño                     1.7 Criptografía, Finanzas, Desarrollo de Videojuegos

--- TOP 5 RECOMENDACIONES (SIMILITUD + CALIDAD) ---
 ID_Recurso                 Titulo             Formato Carrera                                         Etiquetas  Valoracion_Estudiantes  Score_Final
        533   Material_Estudio_533            Artículo  Diseño                            Criptografía, Finanzas                     5.0     0.922664
       5744  Material_Estudio_5744                 PDF  Diseño               Finanzas, Desarrollo de Videojuegos                     5.0     0.884016
       9322  Material_Estudio_9322         Repositorio  Diseño               Finanzas, Desarrollo de Videojuegos                     4.5     0.858310
       6052  Material_Estudio_6052         Repositorio  Diseño Data Science, Finanzas, 